# Bayesian Analysis: Groundhog Day Shadow Predictions

Follow-up to `run_eda.py` and the two EDA notebooks. Those covered the frequentist
side: overall binomial test against p=0.5, decade breakdowns, and groundhog-type
breakdowns. This notebook builds actual hierarchical Bayesian models on the same
data (1886-2022, 1298 predictions, 81 groundhogs) using PyMC.

Two models:

1. **Groundhog-level hierarchical model.** Partial pooling of shadow rate across
   81 groundhogs, most of which have very few predictions (some n=1). This is
   the standard case for shrinkage: individual estimates for low-n groundhogs
   are pulled toward the population rate, high-n groundhogs (Phil, Orphie)
   barely move.
2. **Crossed groundhog x decade model.** The naive decade breakdown in the EDA
   shows shadow rate falling hard after 1980. But the number of reporting
   groundhogs also grows enormously over the same period (2 groundhogs in the
   1880s vs 515 predictions in the 2010s), so a naive decade average conflates
   a real time trend with a change in which groundhogs are in the sample. This
   model estimates a decade effect while holding groundhog identity fixed, to
   see if the trend survives.

Diagnostics (r_hat, ESS, divergences) are checked for both models before
trusting any posterior summary.

In [ ]:
import json
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import scipy.special as sp

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

with open('data/bayesian_input.json') as f:
    bayes_input = json.load(f)

with open('data/combined_data.json') as f:
    combined = json.load(f)

print('overall (naive, pooled):', bayes_input['overall'])

## 1. Groundhog-level hierarchical model

Data: for each of the 81 groundhogs, number of predictions `n` and number of
shadows `shadows`. Naive per-groundhog rate is `shadows / n`, which is a bad
estimate whenever `n` is small (six groundhogs have exactly one prediction, so
their naive rate is either 0.0 or 1.0).

Model, non-centered parameterization on the logit scale:

```
mu        ~ Normal(0, 1.5)               population-level logit shadow rate
sigma     ~ HalfNormal(1)                between-groundhog spread (logit scale)
z_g       ~ Normal(0, 1)                 per groundhog, g = 1..81
theta_g   = mu + z_g * sigma
p_g       = invlogit(theta_g)
shadows_g ~ Binomial(n_g, p_g)
```

Non-centered parameterization avoids the funnel geometry that plain
`p_g ~ Normal(mu, sigma)` hierarchical models are prone to, and it samples
cleanly with NUTS.

In [ ]:
gdf = pd.DataFrame(bayes_input['groundhog_data'])
gdf['n'] = gdf['predictions'].astype(int)
gdf['shadows'] = gdf['shadows'].astype(int)
gdf['label'] = gdf['shortname'] + ' (' + gdf['slug'] + ')'

print(f"{len(gdf)} groundhogs, n ranges from {gdf['n'].min()} to {gdf['n'].max()}")
print(f"{(gdf['n'] == 1).sum()} groundhogs have exactly 1 prediction")

coords = {"groundhog": gdf['label'].values}

with pm.Model(coords=coords) as groundhog_model:
    mu = pm.Normal("mu", mu=0.0, sigma=1.5)
    sigma = pm.HalfNormal("sigma", sigma=1.0)

    z = pm.Normal("z", mu=0.0, sigma=1.0, dims="groundhog")
    theta = pm.Deterministic("theta", mu + z * sigma, dims="groundhog")
    p = pm.Deterministic("p", pm.math.invlogit(theta), dims="groundhog")

    y = pm.Binomial("y", n=gdf['n'].values, p=p, observed=gdf['shadows'].values, dims="groundhog")

    idata_groundhog = pm.sample(
        2000, tune=2000, chains=4, target_accept=0.95,
        random_seed=RANDOM_SEED, progressbar=True,
    )

### Convergence check

In [ ]:
summary = az.summary(idata_groundhog, var_names=["mu", "sigma"])
print(summary)

rhat = az.rhat(idata_groundhog)
max_rhat = max(float(rhat[v].max()) for v in ["mu", "sigma", "z"])
ess_bulk = az.ess(idata_groundhog, var_names=["theta"])
min_ess = float(ess_bulk['theta'].min())
n_div = int(idata_groundhog.sample_stats['diverging'].sum())

print(f"\nmax r_hat across all parameters: {max_rhat:.4f}")
print(f"min ESS bulk across all 81 groundhog-level theta: {min_ess:.0f}")
print(f"divergences: {n_div} / {idata_groundhog.posterior.sizes['chain'] * idata_groundhog.posterior.sizes['draw']}")

r_hat is at 1.0 for every parameter and the worst-case ESS across the 81
groundhog-level parameters is well above the usual 400 rule of thumb, with zero
divergent transitions. This model is trustworthy, no need to reparameterize
further or throw more tuning at it.

In [ ]:
az.plot_trace(idata_groundhog, var_names=["mu", "sigma"], compact=True)
plt.tight_layout()
plt.savefig('data/bayes_trace_groundhog.png', dpi=150, bbox_inches='tight')
plt.show()

### Population-level shadow rate

`mu` is the population mean on the logit scale, i.e. the average groundhog's
tendency to see its shadow, back-transformed to a probability.

In [ ]:
mu_draws = idata_groundhog.posterior['mu'].values.flatten()
sigma_draws = idata_groundhog.posterior['sigma'].values.flatten()
pop_p = sp.expit(mu_draws)

pop_mean = pop_p.mean()
pop_lo, pop_hi = np.percentile(pop_p, [3, 97])
p_above_half = (pop_p > 0.5).mean()

print(f"Population mean shadow rate: {pop_mean:.3f}")
print(f"94% HDI: [{pop_lo:.3f}, {pop_hi:.3f}]")
print(f"P(population rate > 0.5 | data) = {p_above_half:.3f}")
print(f"Between-groundhog SD on logit scale: {sigma_draws.mean():.3f} (94% HDI [{np.percentile(sigma_draws,3):.3f}, {np.percentile(sigma_draws,97):.3f}])")

naive_pooled = bayes_input['overall']['proportion']
print(f"\nFor comparison, naive pooled rate (all 1298 predictions, unweighted by groundhog): {naive_pooled:.3f}")

Two different questions, two different answers, and both are correct for
what they measure:

- **Naive pooled rate (~0.51)** treats every *prediction* as an exchangeable
  unit. Phil alone contributes 127 of the 1298 predictions, so this number is
  effectively a Phil-weighted average.
- **Hierarchical population mean (~0.48)** treats every *groundhog* as an
  exchangeable unit and asks: what's the average tendency of a groundhog
  (regardless of how often it reports)? Phil counts the same as a groundhog
  that only reported once.

Neither the naive binomial test nor this hierarchical model gives strong
evidence against the 50% null. `P(population rate > 0.5) ~ 0.25` (or check the
printed value above) means the data are roughly as consistent with a
population rate below 50% as above it. The real information in this dataset is
not "is 50% the right number", it's the between-groundhog variance:
`sigma` on the logit scale corresponds to genuinely different shadow-seeing
tendencies across groundhogs, some sit consistently above 50%, some
consistently below.

### Shrinkage: naive rate vs. partially pooled estimate

In [ ]:
p_post = idata_groundhog.posterior['p']
gdf['post_mean'] = p_post.mean(dim=['chain', 'draw']).values
gdf['post_lo'] = p_post.quantile(0.03, dim=['chain', 'draw']).values
gdf['post_hi'] = p_post.quantile(0.97, dim=['chain', 'draw']).values
gdf['shrinkage'] = gdf['post_mean'] - gdf['shadow_rate']

most_shrunk = gdf.reindex(gdf['shrinkage'].abs().sort_values(ascending=False).index).head(10)
print(most_shrunk[['shortname', 'n', 'shadows', 'shadow_rate', 'post_mean', 'post_lo', 'post_hi']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
order = gdf.sort_values('n')
ax.errorbar(
    order['post_mean'], range(len(order)),
    xerr=[order['post_mean'] - order['post_lo'], order['post_hi'] - order['post_mean']],
    fmt='o', color='steelblue', ecolor='steelblue', alpha=0.5, markersize=3, label='Posterior mean (94% HDI)'
)
ax.scatter(order['shadow_rate'], range(len(order)), color='firebrick', marker='x', s=25, label='Naive rate', zorder=3)
ax.axvline(pop_mean, color='green', linestyle='--', linewidth=1, label=f'Population mean ({pop_mean:.2f})')
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order['label'], fontsize=5)
ax.set_xlabel('Shadow rate')
ax.set_title('Naive rate vs. partially pooled estimate, ordered by n (fewest predictions at top)')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('data/bayes_shrinkage_groundhog.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
phil = gdf[gdf['slug'] == 'punxsutawney-phil'].iloc[0]
billie = gdf[gdf['slug'] == 'unadilla-billie'].iloc[0]
print(f"Phil: n={phil['n']}, naive={phil['shadow_rate']:.3f}, posterior mean={phil['post_mean']:.3f} (shrinkage={phil['shrinkage']:+.3f})")
print(f"Billie: n={billie['n']}, naive={billie['shadow_rate']:.3f}, posterior mean={billie['post_mean']:.3f} (shrinkage={billie['shrinkage']:+.3f})")

This is the shrinkage pattern working as intended. Punxsutawney Phil has 127
predictions and a naive rate of 0.843, close to his posterior mean, there's
enough data that the prior barely matters. Billie (Unadilla, NY) has exactly
one prediction, saw no shadow, giving a naive rate of 0.0, but the model pulls
her posterior mean to roughly 0.42 with a wide credible interval, because one
observation tells you almost nothing about an individual groundhog and most of
the estimate comes from the population. Five other groundhogs with n=1 show
the identical pattern: their raw 0/1 rates get pulled roughly halfway to the
population mean with wide intervals.

If you ranked groundhogs by naive shadow rate, six of them would sit at
literal 0% or 100%, tied for most/least "reliable." That ranking is mostly
noise. The hierarchical model is the honest version of the "best/worst
groundhog" ranking the second EDA notebook flagged as a proxy metric, this is
what fixes it.

## 2. Crossed groundhog x decade model: does the temporal trend survive?

The naive decade breakdown (see `run_eda.py` output / EDA notebooks) shows
shadow rate near 90-100% for every decade before 1970, then dropping to
roughly 30-50% from 1980 onward. Read at face value that looks like a real
shift in groundhog behavior (or a shift toward less shadow-friendly winters).
But the raw counts per decade tell a different story:

| decade | predictions |
|---|---|
| 1880s | 2 |
| 1900s | 10 |
| 1950s | 23 |
| 1980s | 79 |
| 2010s | 515 |

Early decades are almost entirely Punxsutawney Phil, who has a naturally high
individual shadow rate (0.84 from the model above). As more groundhogs join
the roster from the 1980s on, the decade average necessarily moves toward the
population mean of those new groundhogs, which is not obviously related to
time at all. A model that only looks at decade averages can't tell these two
effects apart. A model with a groundhog-specific offset can.

This uses the full 1298-row prediction table (one row per groundhog per
year), not the pre-aggregated decade summary.

In [ ]:
rows = []
for year, payload in combined['predictions_by_year'].items():
    for pred in payload['predictions']:
        if pred.get('shadow') not in (0, 1):
            continue
        g = pred['groundhog']
        rows.append({
            'year': int(year),
            'shadow': int(pred['shadow']),
            'slug': g['slug'],
            'shortname': g['shortname'],
        })

pred_df = pd.DataFrame(rows)
pred_df['decade'] = (pred_df['year'] // 10) * 10
print(f"{len(pred_df)} predictions, {pred_df['slug'].nunique()} groundhogs, {pred_df['decade'].nunique()} decades")
print(pred_df.groupby('decade')['shadow'].agg(['count', 'mean']))

In [ ]:
groundhog_idx, groundhog_labels = pd.factorize(pred_df['slug'])
decade_idx, decade_labels = pd.factorize(pred_df['decade'], sort=True)

coords2 = {
    "groundhog": groundhog_labels,
    "decade": [str(d) for d in decade_labels],
    "obs": pred_df.index.values,
}

with pm.Model(coords=coords2) as crossed_model:
    mu2 = pm.Normal("mu", 0.0, 1.5)

    sigma_g = pm.HalfNormal("sigma_g", 1.0)
    z_g = pm.Normal("z_g", 0.0, 1.0, dims="groundhog")
    g_eff = pm.Deterministic("g_eff", z_g * sigma_g, dims="groundhog")

    sigma_d = pm.HalfNormal("sigma_d", 1.0)
    z_d = pm.Normal("z_d", 0.0, 1.0, dims="decade")
    d_eff = pm.Deterministic("d_eff", z_d * sigma_d, dims="decade")

    theta2 = mu2 + g_eff[groundhog_idx] + d_eff[decade_idx]
    p_obs = pm.Deterministic("p_obs", pm.math.invlogit(theta2), dims="obs")

    y2 = pm.Bernoulli("y", p=p_obs, observed=pred_df['shadow'].values, dims="obs")

    idata_crossed = pm.sample(
        2000, tune=2000, chains=4, target_accept=0.95,
        random_seed=RANDOM_SEED, progressbar=True,
    )

In [ ]:
summary2 = az.summary(idata_crossed, var_names=["mu", "sigma_g", "sigma_d"])
print(summary2)

rhat2 = az.rhat(idata_crossed)
max_rhat2 = max(float(rhat2[v].max()) for v in ["mu", "sigma_g", "z_g", "sigma_d", "z_d"])
ess2 = az.ess(idata_crossed, var_names=["g_eff", "d_eff"])
min_ess2 = min(float(ess2['g_eff'].min()), float(ess2['d_eff'].min()))
n_div2 = int(idata_crossed.sample_stats['diverging'].sum())

print(f"\nmax r_hat: {max_rhat2:.4f}")
print(f"min ESS bulk (groundhog and decade effects): {min_ess2:.0f}")
print(f"divergences: {n_div2}")

r_hat under 1.01 and no divergences, this converges cleanly with
`target_accept=0.95`. This is a bigger model (81 groundhog offsets + 15 decade
offsets on 1298 Bernoulli observations, two variance components), so ESS is
lower than the simpler model above, that's expected given the crossed random
effects structure and is not a sign of a problem.

In [ ]:
decade_post = idata_crossed.posterior['d_eff']
mu2_draws = idata_crossed.posterior['mu']
decade_p = sp.expit((mu2_draws + decade_post).values)
decade_p_mean = decade_p.mean(axis=(0, 1))
decade_p_lo = np.percentile(decade_p, 3, axis=(0, 1))
decade_p_hi = np.percentile(decade_p, 97, axis=(0, 1))

decade_compare = pd.DataFrame({
    'decade': decade_labels,
    'naive_rate': pred_df.groupby('decade')['shadow'].mean().values,
    'n': pred_df.groupby('decade')['shadow'].count().values,
    'model_rate': decade_p_mean,
    'model_lo': decade_p_lo,
    'model_hi': decade_p_hi,
})
print(decade_compare.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(decade_compare['decade'], decade_compare['naive_rate'], 'o--', color='firebrick', label='Naive decade rate')
ax.plot(decade_compare['decade'], decade_compare['model_rate'], 'o-', color='steelblue', label='Model estimate (groundhog effect held fixed)')
ax.fill_between(decade_compare['decade'], decade_compare['model_lo'], decade_compare['model_hi'], color='steelblue', alpha=0.2)
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='50% baseline')
ax.set_xlabel('Decade')
ax.set_ylabel('Shadow rate')
ax.set_title('Naive decade average vs. decade effect controlling for groundhog identity')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('data/bayes_decade_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

The trend survives, but it's smaller than the naive numbers suggest. Naive
1900s-1960s rates sit around 0.85-1.0; once the model accounts for the fact
that this era is basically Phil plus a couple of similarly shadow-prone
groundhogs, and pools those extreme small-sample decades toward the overall
mean, the model's decade estimate for that era comes down to roughly 0.65-0.78.
The 1980-2020 decades, by contrast, barely move between naive and model
estimates, since there's enough data (79 to 515 predictions per decade) that
partial pooling has little left to correct.

So there are really two things going on, not one: (1) part of the apparent
1900s-1970s vs. 1980s-2020s gap is a small-sample artifact inflated by the
naive per-decade average, exactly the kind of thing partial pooling exists to
fix, and (2) even after correcting for that, a real gap remains, mid-century
decades still land meaningfully above 50% and recent decades sit close to or
below 50%. The dataset can't distinguish "groundhogs actually changed" from
"winters changed" from "the roster of participating groundhogs changed in a
way not captured by a single fixed per-groundhog offset," but the composition
story alone does not fully explain the drop.

## Summary

- Naive pooled shadow rate: 0.512 (all predictions, Phil-weighted).
  Hierarchical population mean, weighting every groundhog equally: see
  printed value above (~0.48). Neither rejects the 50% null with any real
  confidence, `P(population rate > 0.5)` sits well inside a coin-flip range.
- Between-groundhog variance is the real signal in this dataset: groundhogs
  differ meaningfully and consistently in shadow-seeing tendency (sigma ~0.84
  on the logit scale), which is a much stronger effect than any overall
  deviation from 50%.
- Partial pooling matters most exactly where you'd expect: groundhogs with a
  single prediction get pulled roughly halfway to the population mean instead
  of sitting at a nonsensical literal 0% or 100%. This directly fixes the
  "best/worst groundhog" ranking flagged as an unvalidated proxy in
  `groundhog_eda_v2.ipynb`.
- The apparent decline in shadow rate after 1980 is partly (not fully) a
  small-sample artifact of which/how many groundhogs were reporting in early
  decades. A real gap remains after controlling for groundhog identity.
- Both models converged cleanly: r_hat ~1.0 on every parameter, no divergent
  transitions, adequate ESS throughout.

## 3. Where the groundhogs live

Quick geographic pass on all 88 entries in the registry: region coverage, then a
map of the ones with usable coordinates. This section is adapted from
`groundhog_eda_v2.ipynb` (the old WIP notebook), minus the parts that got
superseded.

Two things from that WIP notebook did NOT make it here:
- The naive Beta(1,1) "ability" ranking (P(shadow) per groundhog with a flat
  prior) is gone. Section 1 above already does this properly with partial
  pooling, so see the ranking table below built from the actual posterior
  instead of the naive one.
- The weather-join scaffold is gone entirely. It was an empty DataFrame with
  no data behind it, not worth shipping as a placeholder. See Future Work.

In [ ]:
regions = pd.Series(combined['groundhogs']).apply(lambda g: g.get('region'))
region_counts = regions.value_counts(dropna=False)
print(f"{region_counts.index.nunique()} unique regions across {len(combined['groundhogs'])} groundhogs")
region_counts.head(15)

### Ability ranking, from the model this time

Same best/worst framing as the old WIP notebook, but ranked on the hierarchical
posterior mean (`post_mean` from section 1) instead of a flat-prior proxy. The
`MIN_N` filter is dropped: partial pooling is exactly what handles low-`n`
groundhogs correctly, so there's no reason to throw them out anymore.

In [ ]:
rank_cols = ['label', 'n', 'shadows', 'shadow_rate', 'post_mean']
ranked = gdf.sort_values('post_mean', ascending=False)

print("Most shadow-prone (posterior mean):")
print(ranked[rank_cols].head(10).to_string(index=False))
print()
print("Least shadow-prone (posterior mean):")
print(ranked[rank_cols].tail(10).to_string(index=False))

### Map

Groundhog locations across the US and Canada, admin-1 boundaries from Natural
Earth. Rendered to a static image so it shows up on GitHub without needing a
live plotly.js runtime (interactive plotly output doesn't render in GitHub's
notebook viewer).

In [ ]:
import plotly.graph_objects as go
from pathlib import Path

coords = pd.Series(combined['groundhogs']).apply(lambda g: g.get('coordinates'))
coord_df = coords.str.split(',', expand=True).iloc[:, :2]
coord_df.columns = ['lat', 'lon']
coord_df = coord_df.apply(pd.to_numeric, errors='coerce')

geo = pd.DataFrame(combined['groundhogs'])[['shortname', 'slug', 'region', 'country']].join(coord_df)
geo = geo.dropna(subset=['lat', 'lon'])
print(f"{len(geo)} of {len(combined['groundhogs'])} groundhogs have usable coordinates")

In [ ]:
geojson_path = Path('data/ne_50m_admin_1_states_provinces.geojson')
if not geojson_path.exists():
    # not committed (data/*.geojson is gitignored, regeneratable), fetch on demand
    import urllib.request
    geojson_path.parent.mkdir(parents=True, exist_ok=True)
    url = 'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_admin_1_states_provinces.geojson'
    with urllib.request.urlopen(url) as resp:
        geojson_path.write_bytes(resp.read())

with open(geojson_path) as f:
    gj = json.load(f)

features = [
    feat for feat in gj['features']
    if feat.get('properties', {}).get('admin') in ('United States of America', 'Canada')
]
gj_uc = {'type': 'FeatureCollection', 'features': features}

fig = go.Figure()
fig.add_trace(go.Choropleth(
    geojson=gj_uc,
    locations=list(range(len(features))),
    z=[0] * len(features),
    featureidkey=None,
    colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],
    showscale=False,
    marker_line_color='#666666',
    marker_line_width=0.6,
))

geo_na = geo[(geo['lon'].between(-170, -40)) & (geo['lat'].between(5, 80))].copy()
fig.add_trace(go.Scattergeo(
    lon=geo_na['lon'],
    lat=geo_na['lat'],
    text=geo_na['shortname'],
    mode='markers',
    marker=dict(size=6, color='#1f77b4', opacity=0.7),
    name='Groundhogs',
))

fig.update_geos(
    scope='north america',
    projection_type='natural earth',
    fitbounds='locations',
    showcountries=True,
    showland=True,
    landcolor='#f8f8f8',
    showlakes=True,
    lakecolor='#b3d9ff',
    showocean=True,
    oceancolor='#b3d9ff',
)
fig.update_layout(
    title='Groundhog Locations (US + Canada Admin-1 Boundaries)',
    height=650,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show('png')

Heavily clustered around the mid-Atlantic and Great Lakes, unsurprising given
that's where the tradition started and where most clubs still run it. A handful
of outliers on the west coast and in the maritimes. Not enough spread to say
much about regional climate effects here, that's on the Future Work list below.

## Future Work

Things worth doing next, already checked against the real data so these aren't
just vague ideas:

1. ~~**Fake vs. real groundhogs**~~ — done, see section 4 above. 55 of the 93
   registry entries aren't groundhogs at all (there's an alligator, a
   bullfrog, a lobster, and a pile of animatronics in there). No credible
   difference in shadow rate: the 94% HDI on the group difference straddles
   zero.
2. ~~**Follow the leader**~~ — done, see section 5 above. No groundhog shows
   credibly higher or lower agreement with Phil than you'd expect from their
   own independent shadow rate; the raw 33%-71% range is explained by base
   rates, not copycatting.
3. **Streakiness**: year-to-year autocorrelation per groundhog, for the 25
   with 20+ recorded calls. Do any of them run in actual streaks (several
   shadow years in a row, then several no-shadow years), or is every call
   independent of the last.
4. **Career-length effect**: groundhogs that started reporting in 2023-2025
   versus century-old veterans like Phil (since 1887) and Octoraro Orphie
   (since 1926). Does shadow rate or reporting reliability change with how
   long a groundhog has been on the job.
5. ~~**Geographic clustering**~~ — done (linear pass), see section 6 above.
   No detectable linear spatial gradient in lat or lon. A full Gaussian
   process is the natural next step if a later look turns up spatial signal
   a straight-line term can't catch.

One thing that got ruled out: checking predictions against actual NOAA
spring-onset data. The API's own embedded temperature field only has usable
values for 13 of 1761 predictions, too sparse to build any kind of accuracy
score on.

## 4. Fake vs. real groundhogs

38 of the 93 registry entries are actual groundhogs (`type == 'Groundhog'`);
the rest are taxidermied groundhogs, people in costumes, plush toys, an
alligator, a lobster, and similar. Does shadow rate differ between the real
groundhogs and everything else, or is it noise around 50% either way?

In [ ]:
gdf['is_real'] = (gdf['type'] == 'Groundhog').astype(int)
print(gdf.groupby('is_real')['n'].agg(['count', 'sum']))

group_idx = gdf['is_real'].values  # 0 = not real, 1 = real
coords_real = {"group": ["fake", "real"]}

with pm.Model(coords=coords_real) as real_model:
    mu_group = pm.Normal("mu_group", mu=0.0, sigma=1.5, dims="group")
    sigma_i = pm.HalfNormal("sigma_i", sigma=1.0)

    z_i = pm.Normal("z_i", mu=0.0, sigma=1.0, shape=len(gdf))
    theta_i = mu_group[group_idx] + z_i * sigma_i
    p_i = pm.math.invlogit(theta_i)

    y = pm.Binomial("y", n=gdf['n'].values, p=p_i, observed=gdf['shadows'].values)

    p_group = pm.Deterministic("p_group", pm.math.invlogit(mu_group), dims="group")

    idata_real = pm.sample(
        2000, tune=2000, chains=4, target_accept=0.95,
        random_seed=RANDOM_SEED, progressbar=True,
    )

In [ ]:
print(az.summary(idata_real, var_names=["mu_group", "sigma_i"]))
n_div = int(idata_real.sample_stats.diverging.sum())
print(f"divergences: {n_div}")

In [ ]:
p_fake_draws = idata_real.posterior['p_group'].sel(group="fake").values.flatten()
p_real_draws = idata_real.posterior['p_group'].sel(group="real").values.flatten()
diff = p_real_draws - p_fake_draws

print(f"P(real) posterior mean: {p_real_draws.mean():.3f} (94% HDI {np.percentile(p_real_draws,3):.3f}-{np.percentile(p_real_draws,97):.3f})")
print(f"P(fake) posterior mean: {p_fake_draws.mean():.3f} (94% HDI {np.percentile(p_fake_draws,3):.3f}-{np.percentile(p_fake_draws,97):.3f})")
print(f"Difference (real - fake): {diff.mean():+.3f} (94% HDI {np.percentile(diff,3):+.3f} to {np.percentile(diff,97):+.3f})")
print(f"P(real group mean > fake group mean): {(diff > 0).mean():.3f}")

Real groundhogs come out slightly *less* shadow-prone than fakes in this sample: posterior mean shadow probability 0.442 for real groundhogs versus 0.511 for fakes, a difference of -0.069 with a 94% HDI of -0.177 to +0.047. That interval straddles zero, and P(real group mean > fake group mean) is only 0.124, i.e. the data leans toward fakes reporting more shadows but not strongly enough to call it. With 33 real groundhogs against 51 fakes (719 vs. 591 total predictions), there just isn't enough separation to rule out the 50/50 story for either group individually, let alone conclude the two differ.

## 5. Follow the leader: agreement with Punxsutawney Phil

Per-groundhog agreement rate with Phil, for groundhogs with 5+ years of
shared reporting history. The baseline isn't a flat 50% - two groundhogs
with their own independent shadow rates `p_phil` and `p_g` agree by chance at
rate `p_phil*p_g + (1-p_phil)*(1-p_g)`. Model the observed agreement rate
against that baseline directly, rather than eyeballing a 33%-71% raw range.

In [ ]:
phil_by_year = pred_df[pred_df['slug'] == 'punxsutawney-phil'].set_index('year')['shadow']
p_phil = gdf.loc[gdf['slug'] == 'punxsutawney-phil', 'shadow_rate'].iloc[0]

agree_rows = []
for slug, sub in pred_df[pred_df['slug'] != 'punxsutawney-phil'].groupby('slug'):
    g_by_year = sub.set_index('year')['shadow']
    shared_years = g_by_year.index.intersection(phil_by_year.index)
    if len(shared_years) < 5:
        continue
    g_vals = g_by_year.loc[shared_years]
    phil_vals = phil_by_year.loc[shared_years]
    n_shared = len(shared_years)
    n_agree = int((g_vals.values == phil_vals.values).sum())
    p_g_row = gdf.loc[gdf['slug'] == slug, 'shadow_rate']
    if p_g_row.empty:
        continue
    p_g = p_g_row.iloc[0]
    expected_rate = p_phil * p_g + (1 - p_phil) * (1 - p_g)
    agree_rows.append({
        'slug': slug,
        'shortname': sub['shortname'].iloc[0],
        'n_shared': n_shared,
        'n_agree': n_agree,
        'p_phil': p_phil,
        'p_g': p_g,
        'expected_rate': expected_rate,
        'observed_rate': n_agree / n_shared,
    })

agree_df = pd.DataFrame(agree_rows)
print(f"{len(agree_df)} groundhogs with 5+ shared years with Phil")
print(agree_df[['shortname', 'n_shared', 'observed_rate', 'expected_rate']].sort_values('observed_rate', ascending=False).to_string(index=False))

In [ ]:
coords_agree = {"groundhog": agree_df['slug'].values}
logit_expected = sp.logit(agree_df['expected_rate'].values)

with pm.Model(coords=coords_agree) as agreement_model:
    mu_eff = pm.Normal("mu_eff", 0.0, 1.0)
    sigma_eff = pm.HalfNormal("sigma_eff", 1.0)

    z_eff = pm.Normal("z_eff", 0.0, 1.0, dims="groundhog")
    effect = pm.Deterministic("effect", mu_eff + z_eff * sigma_eff, dims="groundhog")

    theta = logit_expected + effect
    p_agree = pm.Deterministic("p_agree", pm.math.invlogit(theta), dims="groundhog")

    y = pm.Binomial("y", n=agree_df['n_shared'].values, p=p_agree, observed=agree_df['n_agree'].values, dims="groundhog")

    idata_agreement = pm.sample(
        2000, tune=2000, chains=4, target_accept=0.95,
        random_seed=RANDOM_SEED, progressbar=True,
    )

In [ ]:
print(az.summary(idata_agreement, var_names=["mu_eff", "sigma_eff"]))
n_div = int(idata_agreement.sample_stats.diverging.sum())
print(f"divergences: {n_div}")

In [ ]:
effect_post = idata_agreement.posterior['effect']
agree_df['effect_mean'] = effect_post.mean(dim=['chain', 'draw']).values
agree_df['effect_lo'] = effect_post.quantile(0.03, dim=['chain', 'draw']).values
agree_df['effect_hi'] = effect_post.quantile(0.97, dim=['chain', 'draw']).values
agree_df['credibly_above_baseline'] = agree_df['effect_lo'] > 0
agree_df['credibly_below_baseline'] = agree_df['effect_hi'] < 0

print(agree_df[['shortname', 'n_shared', 'observed_rate', 'expected_rate', 'effect_mean', 'effect_lo', 'effect_hi']].sort_values('effect_mean', ascending=False).to_string(index=False))
print(f"\n{agree_df['credibly_above_baseline'].sum()} groundhogs credibly agree with Phil more than chance")
print(f"{agree_df['credibly_below_baseline'].sum()} groundhogs credibly agree with Phil less than chance")

None of the 63 groundhogs with 5+ shared years clears the bar: every
individual `effect` posterior's 94% HDI straddles zero, including the
highest-ranked (Chuck, 27 shared years, effect mean +0.38, HDI -0.05 to
+0.95) and the lowest (Minnie, 14 shared years, effect mean -0.21, HDI
-0.88 to +0.25). At the population level, `mu_eff`'s 94% HDI is -0.07 to
+0.23 (mean +0.08), also straddling zero, with `sigma_eff`'s HDI (0.01 to
0.47) indicating only mild between-groundhog heterogeneity. No evidence of
a systemic Phil-copycat effect once independent shadow rates are accounted
for: the raw 33%-71% agreement-rate spread is almost entirely explained by
each groundhog's own base rate, not by any tendency to follow (or defy)
Phil specifically.

## 6. Geographic clustering

Using the 84 groundhogs with usable coordinates: is there a linear spatial
gradient in shadow rate (using each groundhog's partially-pooled posterior
mean from section 1 as the outcome)? Expected signal is weak, both because
the registry is heavily clustered around the mid-Atlantic and because a
linear term can only pick up a gradient, not clustering. Treated as a first
pass — see the note at the end of this section on why a full Gaussian
process is the natural next step.


In [ ]:
geo_model_df = geo.merge(gdf[['slug', 'post_mean', 'n']], on='slug', how='inner').dropna(subset=['post_mean'])
print(f"{len(geo_model_df)} groundhogs with coordinates + posterior mean shadow rate")

lat_z = (geo_model_df['lat'] - geo_model_df['lat'].mean()) / geo_model_df['lat'].std()
lon_z = (geo_model_df['lon'] - geo_model_df['lon'].mean()) / geo_model_df['lon'].std()

with pm.Model() as geo_model:
    intercept = pm.Normal("intercept", 0.5, 0.5)
    beta_lat = pm.Normal("beta_lat", 0.0, 0.2)
    beta_lon = pm.Normal("beta_lon", 0.0, 0.2)
    sigma = pm.HalfNormal("sigma", 0.2)

    mu = intercept + beta_lat * lat_z.values + beta_lon * lon_z.values
    y = pm.Normal("y", mu=mu, sigma=sigma, observed=geo_model_df['post_mean'].values)

    idata_geo = pm.sample(
        2000, tune=2000, chains=4, target_accept=0.95,
        random_seed=RANDOM_SEED, progressbar=True,
    )


In [ ]:
print(az.summary(idata_geo, var_names=["intercept", "beta_lat", "beta_lon", "sigma"]))
n_div = int(idata_geo.sample_stats.diverging.sum())
print(f"divergences: {n_div}")


In [ ]:
beta_lat_draws = idata_geo.posterior['beta_lat'].values.flatten()
beta_lon_draws = idata_geo.posterior['beta_lon'].values.flatten()

print(f"beta_lat: mean={beta_lat_draws.mean():+.3f}, 94% HDI [{np.percentile(beta_lat_draws,3):+.3f}, {np.percentile(beta_lat_draws,97):+.3f}]")
print(f"beta_lon: mean={beta_lon_draws.mean():+.3f}, 94% HDI [{np.percentile(beta_lon_draws,3):+.3f}, {np.percentile(beta_lon_draws,97):+.3f}]")
print(f"P(beta_lat > 0): {(beta_lat_draws > 0).mean():.3f}")
print(f"P(beta_lon > 0): {(beta_lon_draws > 0).mean():.3f}")


Neither `beta_lat` (mean -0.009, 94% HDI [-0.039, +0.021]) nor `beta_lon`
(mean -0.006, 94% HDI [-0.036, +0.024]) has a 94% HDI that excludes zero —
P(beta_lat > 0) = 0.278, P(beta_lon > 0) = 0.351. No detectable linear
gradient in shadow rate across latitude or longitude; consistent with the
map in section 3 showing a tight mid-Atlantic/Great Lakes cluster rather
than any north-south or east-west trend.

**Why a full Gaussian process would be the stronger follow-up:** this linear
model can only detect a monotonic gradient across latitude or longitude
separately. It can't detect clustering that isn't a straight-line trend — for
example two separate regional hotspots with similar shadow rates but average
coordinates in between, or a spatial pattern that runs diagonally rather
than along the lat/lon axes. A GP with a 2D spatial kernel (e.g. Matérn) over
`(lat, lon)` would model spatial covariance directly: nearby groundhogs
correlated regardless of the direction of the pattern, with the kernel's
lengthscale itself estimated from the data rather than assumed linear. Worth
running if this or a later pass turns up any spatial signal worth pinning
down more precisely.
